<a href="https://colab.research.google.com/github/chantika-maharani/ProyekAkhir_5C_Estimation-of-Obesity-Levels/blob/main/02_praproses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laporan Praproses Data untuk Algoritma K-Nearest Neighbors (KNN)

Praproses data merupakan tahapan krusial sebelum menerapkan algoritma KNN. Karena KNN menggunakan perhitungan jarak geometris (seperti *Euclidean Distance*), performa model sangat dipengaruhi oleh perbedaan skala fitur dan keberadaan data kategorikal. Berikut adalah tahapan praproses terstruktur yang dirancang untuk menjaga integritas data tanpa menyebabkan kebocoran informasi (*data leakage*).

### Langkah 1: Memisahkan Fitur (X) dan Target (y)
Kami memisahkan variabel independen (fitur/X) dengan variabel dependen (target/y). Dalam kasus ini, target prediktor berada di kolom terakhir dataset.

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

df = pd.read_csv('/content/data_bersih (1).csv')
target_column = df.columns[-1]

X = df.drop(columns=[target_column])
y = df[target_column]

### Langkah 2: Identifikasi Tipe Atribut (Numerik vs Kategorikal)
Atribut numerik dan kategorikal memerlukan penanganan yang berbeda. Kami mengelompokkan jenis kolom ini secara otomatis berdasarkan tipe datanya.

In [20]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Fitur Numerik: {num_cols}")
print(f"Fitur Kategorikal: {cat_cols}")

Fitur Numerik: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Fitur Kategorikal: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


### Langkah 3: Konstruksi Pipeline Praproses
Untuk mempermudah standarisasi, kami merancang `Pipeline` menggunakan `ColumnTransformer`:
1. **Fitur Numerik**: Diimputasi menggunakan nilai *median* untuk menangani data kosong tanpa sensitif terhadap pencilan (outliers), kemudian distandarisasikan menggunakan `StandardScaler` agar memiliki rata-rata 0 dan varians 1.
2. **Fitur Kategorikal**: Diimputasi menggunakan nilai modus (*most frequent*), lalu dikonversi menjadi biner menggunakan `OneHotEncoder` agar metrik jarak KNN dapat memprosesnya.

In [21]:
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

### Langkah 4: Pembagian dan Transformasi Data (Split & Fit)
Data dibagi menjadi data latih (80%) dan data uji (20%). Fungsi `.fit_transform()` hanya diaplikasikan pada data latih untuk menghindari kebocoran informasi (*data leakage*), sedangkan data uji hanya disesuaikan menggunakan `.transform()`.

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)

print(f"Dimensi data latih hasil praproses: {X_train_final.shape}")
print(f"Dimensi data uji hasil praproses: {X_test_final.shape}")

Dimensi data latih hasil praproses: (1669, 31)
Dimensi data uji hasil praproses: (418, 31)
